# Example: Building an Autotrader Validation Report
In this example, we convert a paired-path strategy comparison into explicit deployment gates and a compact risk-control configuration.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
>
> * __Construct paired validation evidence:__ Compare adaptive and frozen policies on common simulated paths.
> * __Apply pass/fail gates:__ Evaluate reward, drawdown, failure, benchmark, and NPV conditions.
> * __Separate evidence from authorization:__ Explain why passing a simplified gate is necessary but not sufficient for live deployment.

Let's replace a vague claim that the strategy “looks good” with an auditable decision table.
___


## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading the packages used in this example.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates `Include.jl` in the notebook's global scope. The file activates the course environment, defines notebook-relative paths, and loads the required packages.

Let's set up the code environment:

The reusable portfolio algorithms in this example are provided by the local [`VLQuantitativeFinancePackage.jl`](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/) package.


In [ ]:
include(joinpath(@__DIR__, "Include.jl"));


For additional information, see the [Julia documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

___


## Task 1: Generate Paired Validation Results
We generate paired terminal wealth observations for frozen and adaptive configurations. A shared market shock creates dependence between the two outcomes, so the paired difference is more informative than two unrelated samples.


In [ ]:
Random.seed!(5660);
number_of_paths = 1500;
market_shock = rand(Normal(0.04, 0.20), number_of_paths);
frozen_terminal = exp.(market_shock .+ rand(Normal(-0.01, 0.10), number_of_paths));
adaptive_terminal = exp.(market_shock .+ rand(Normal(0.015, 0.09), number_of_paths));

frozen_drawdown = clamp.(rand(Normal(0.19, 0.06), number_of_paths), 0, 1);
adaptive_drawdown = clamp.(rand(Normal(0.15, 0.05), number_of_paths), 0, 1);
adaptive_turnover = clamp.(rand(Normal(1.8, 0.45), number_of_paths), 0, Inf);

risk_free_yield = 0.04;
discount_factor = exp(-risk_free_yield);
adaptive_npv = discount_factor .* adaptive_terminal .- 1.0;
paired_excess = adaptive_terminal .- frozen_terminal;


## Task 2: Compute the Validation Metrics
The gates use metrics with distinct meanings: median paired improvement, median NPV, failure probability, tail drawdown, and turnover. Thresholds are declared before reading the pass/fail result.


In [ ]:
metrics = Dict(
    "median_paired_excess" => median(paired_excess),
    "median_npv" => median(adaptive_npv),
    "failure_probability" => mean(adaptive_terminal .< 1.0),
    "drawdown_95" => quantile(adaptive_drawdown, 0.95),
    "median_turnover" => median(adaptive_turnover),
);

limits = Dict(
    "median_paired_excess" => (operator=:minimum, value=0.0),
    "median_npv" => (operator=:minimum, value=0.0),
    "failure_probability" => (operator=:maximum, value=0.42),
    "drawdown_95" => (operator=:maximum, value=0.25),
    "median_turnover" => (operator=:maximum, value=2.25),
);


## Task 3: Produce the Gate Report
Each row records the observed value, declared limit, direction of the test, and result. The overall decision is `PASS` only if every required gate passes.


In [ ]:
gate_result = evaluate_validation_gates(metrics, limits);
report = DataFrame(gate_result.report);
overall = gate_result.overall;
pretty_table(report; table_format=TextTableFormat(borders=text_table_borders__simple));
println("Overall classroom validation decision: $(overall)")


In [ ]:
risk_control_configuration = (
    validation_decision=overall,
    max_position_weight=0.30,
    max_one_way_turnover=0.20,
    drawdown_escalation=0.10,
    news_severity_escalation=0.75,
    unattended_live_trading=false,
);
risk_control_configuration


## Summary
This example converted a strategy comparison into a reproducible validation decision.

> __Key Takeaways:__
>
> * __Thresholds should precede the verdict:__ Choosing gates after inspecting results invites confirmation bias.
> * __Paired comparisons reduce ambiguity:__ Common paths isolate policy differences from scenario differences.
> * __A classroom pass is not production approval:__ Data provenance, software testing, operational controls, legal review, and human accountability remain outside this simplified report.

The exported configuration supplies limits for the captured-event operations example.
___

## Disclaimer and Risks
This validation report is a teaching artifact. It is not a regulatory, compliance, or investment-approval process.
